In [1]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# 1. Load your dataset directly from the CSV file
df = pd.read_csv("telecom_customer_churn_feature_engineering.csv")

# Let's use 'monthly_bill' as our target column for handling outliers
print("Original Max Bill:", df['monthly_bill'].max())
print("-" * 50)


# ==========================================
# Technique 1: Clipping
# ==========================================
# Clipping caps the extreme values at a specific upper and lower threshold.
# Here, we cap the values at the 5th and 95th percentiles.

lower_limit = df["monthly_bill"].quantile(0.05)
upper_limit = df["monthly_bill"].quantile(0.95)

# Create a new column with the clipped values
df["bill_clipped"] = df["monthly_bill"].clip(lower=lower_limit, upper=upper_limit)

print("Max Bill after Clipping (95th percentile):", df['bill_clipped'].max())


# ==========================================
# Technique 2: Winsorization
# ==========================================
# Winsorization replaces extreme data values with less extreme values (percentiles).
# The limits=[0.05, 0.05] means we cap the bottom 5% and top 5%.

# Note: winsorize returns a masked array, so we assign it back to a new pandas column
df["bill_winsorized"] = winsorize(df["monthly_bill"], limits=[0.05, 0.05])

print("Max Bill after Winsorization (top 5% capped):", df['bill_winsorized'].max())


# ==========================================
# Technique 3: Removal
# ==========================================
# Removal drops the rows that contain outlier values completely.
# We'll use the standard Interquartile Range (IQR) method to define boundaries.

Q1 = df["monthly_bill"].quantile(0.25)
Q3 = df["monthly_bill"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Create a new dataframe keeping only the rows within the acceptable boundaries
df_removed = df[(df["monthly_bill"] >= lower_bound) & (df["monthly_bill"] <= upper_bound)].copy()

print(f"Original row count: {len(df)}")
print(f"Row count after IQR Removal: {len(df_removed)}")
print("Max Bill in the new dataset after removal:", df_removed['monthly_bill'].max())
print("-" * 50)

# View a sample of the transformed data side-by-side to compare
print("\nComparison of original, clipped, and winsorized features:")
print(df[["monthly_bill", "bill_clipped", "bill_winsorized"]].sort_values(by="monthly_bill", ascending=False).head())

Original Max Bill: 36361.0
--------------------------------------------------
Max Bill after Clipping (95th percentile): 13755.199999999997
Max Bill after Winsorization (top 5% capped): 13752.0
Original row count: 1200
Row count after IQR Removal: 1200
Max Bill in the new dataset after removal: 13752.0
--------------------------------------------------

Comparison of original, clipped, and winsorized features:
     monthly_bill  bill_clipped  bill_winsorized
12        13752.0       13755.2          13752.0
463       13752.0       13755.2          13752.0
484       13752.0       13755.2          13752.0
536       13752.0       13755.2          13752.0
497       13752.0       13755.2          13752.0
